# YOLO 2회차 추가 실습 — 오토 라벨링 공장

현업 프로젝트에서 가장 비싸고 오래 걸리는 일은 모델 학습이 아니라 **라벨링**입니다. 이미지 수천 장에 박스를 하나하나 그려야 하니까요.
이 실습에서는 '말로 찾는 탐지기' YOLOE에게 라벨링을 맡겨 **자동 라벨**을 만들고, 그 라벨로 학습한 모델이 **사람이 그린 라벨로 학습한 모델**과 비교해 얼마나 쓸 만한지 직접 확인합니다.
마지막에는 자동 라벨 중 애매한 것만 사람이 고치는 **하이브리드 방식**의 효과와 비용까지 따져 봅니다.

| 단계 | 만드는 것 | 결과물 |
|---|---|---|
| Step 1 | 목표 정하기 — 안전모·조끼·사람 3종 데이터 만들기 | 사람 라벨 데이터셋 |
| Step 2 | 자동 라벨링 — YOLOE로 학습 이미지 전체에 라벨 달기 | 예측 캐시 |
| Step 3 | 라벨 품질 검사 — 사람 라벨과 비교, 신뢰도 임계값 고르기 | 자동 라벨 데이터셋 |
| Step 4 | 공장 가동 — 사람 라벨 모델 vs 자동 라벨 모델 학습·평가 | 성능 비교표 |
| Step 5 | 하이브리드 검수와 비용 계산 | `label_battle.csv`, `label_cost.csv`, `label_report.png` |

**데이터셋**: Ultralytics 공개 *Construction-PPE* (건설 현장 작업자 사진, 학습 1,132장 · 검증 143장 · 테스트 141장)

**실행 환경**: Google Colab, 런타임 유형 **T4 GPU** (학습이 세 번 들어가므로 GPU 필수, 전체 약 30~40분)

## 0. 준비

In [ ]:
!pip -q install ultralytics
!pip -q install git+https://github.com/ultralytics/CLIP.git

In [ ]:
import os, json, time, shutil
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from ultralytics import YOLO, YOLOE
from ultralytics.data.utils import check_det_dataset

print('GPU 사용 가능:', torch.cuda.is_available())

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    drive.mount('/content/drive')
    SAVE_DIR = Path('/content/drive/MyDrive/yolo_day2/autolabel')
else:
    SAVE_DIR = Path('outputs/autolabel')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
WORK = Path('ppe3')            # 이 실습에서 만드는 데이터셋 폴더 (런타임 로컬 디스크)
print('결과 저장 위치:', SAVE_DIR.resolve())

def save_out(path):
    dst = SAVE_DIR / Path(path).name
    if Path(path).resolve() != dst.resolve():
        shutil.copy(path, dst)
    return dst

def show_grid(imgs, titles=None, cols=3, size=4):
    rows = (len(imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows))
    for ax, i in zip(np.array(axes).ravel(), range(rows * cols)):
        ax.axis('off')
        if i < len(imgs):
            ax.imshow(imgs[i][..., ::-1])
            if titles:
                ax.set_title(titles[i], fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
# 학습 설정 — 시간이 부족하면 QUICK_RUN = True
QUICK_RUN = False
CFG = dict(epochs=30, imgsz=640, batch=16) if not QUICK_RUN else dict(epochs=5, imgsz=416, batch=16)
print(CFG)

---
## Step 1. 목표 정하기 — 안전모·조끼·사람

원본 데이터셋에는 11개 클래스가 있지만, 이번 공장에서는 현장에서 가장 많이 쓰는 **3종**만 다룹니다. 원본 번호를 새 번호로 바꾼 '사람 라벨 데이터셋'을 만들어 두고, 이것을 비교 기준(정답)으로 씁니다.

| 새 번호 | 이름 | 원본 번호·이름 | YOLOE에게 줄 단어 |
|---|---|---|---|
| 0 | helmet | 0 · helmet | `helmet` |
| 1 | vest | 2 · vest | `safety vest` |
| 2 | person | 6 · Person | `person` |

In [ ]:
PPE = check_det_dataset('construction-ppe.yaml')
SRC = Path(PPE['path'])
print('원본 클래스:', PPE['names'])

TARGET = {0: 'helmet', 1: 'vest', 2: 'person'}
SRC_TO_NEW = {0: 0, 2: 1, 6: 2}
PROMPTS = {0: 'helmet', 1: 'safety vest', 2: 'person'}
SPLITS = ['train', 'val', 'test']

def read_labels(path, mapping=None):
    """YOLO 라벨 파일 → [(클래스, cx, cy, w, h)]. mapping이 있으면 번호를 바꾸고 나머지는 버림"""
    out = []
    if Path(path).exists():
        for line in Path(path).read_text().strip().splitlines():
            c, *v = line.split()
            c = int(c)
            if mapping is not None:
                if c not in mapping:
                    continue
                c = mapping[c]
            out.append((c, *map(float, v[:4])))
    return out

def write_labels(path, labels):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(''.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n' for c, cx, cy, w, h in labels))

def link_images(dst_root):
    """원본 이미지를 복사하지 않고 바로가기(심볼릭 링크)로 연결"""
    for split in SPLITS:
        d = dst_root / 'images' / split
        d.mkdir(parents=True, exist_ok=True)
        for p in (SRC / 'images' / split).iterdir():
            if not (d / p.name).exists():
                os.symlink(p.resolve(), d / p.name)

def write_yaml(root):
    cfg = {'path': str(root.resolve()), 'train': 'images/train', 'val': 'images/val',
           'test': 'images/test', 'names': TARGET}
    yml = root / 'data.yaml'
    yml.write_text(json.dumps(cfg))
    return str(yml)

HUMAN = WORK / 'human'
link_images(HUMAN)
for split in SPLITS:
    for p in (SRC / 'images' / split).iterdir():
        labels = read_labels(SRC / 'labels' / split / (p.stem + '.txt'), SRC_TO_NEW)
        write_labels(HUMAN / 'labels' / split / (p.stem + '.txt'), labels)
HUMAN_YAML = write_yaml(HUMAN)

TRAIN_IMAGES = sorted((HUMAN / 'images' / 'train').iterdir())
cnt = Counter(c for p in TRAIN_IMAGES for c, *_ in read_labels(HUMAN / 'labels' / 'train' / (p.stem + '.txt')))
print('학습 이미지:', len(TRAIN_IMAGES), '장')
pd.DataFrame([{'class': TARGET[k], 'human_boxes': cnt[k]} for k in TARGET])

---
## Step 2. 자동 라벨링

YOLOE에게 세 단어를 주고 학습 이미지 1,132장을 모두 예측시킵니다. 이때 **신뢰도 0.05라는 아주 낮은 기준**으로 모든 후보를 저장해 둡니다(예측 캐시).
어느 신뢰도부터 라벨로 인정할지는 Step 3에서 정하므로, 캐시가 있으면 기준을 바꿀 때마다 모델을 다시 돌릴 필요가 없습니다.

예측 결과는 픽셀 좌표이므로, YOLO 라벨 형식(이미지 크기로 나눈 0~1 사이의 중심·너비·높이)으로 바꿔 저장합니다.

In [ ]:
labeler = YOLOE('yoloe-26s-seg.pt')
labeler.set_classes([PROMPTS[k] for k in sorted(PROMPTS)])     # 결과 클래스 번호 = PROMPTS 순서 = 새 번호

t0 = time.time()
CACHE = {}
for p in TRAIN_IMAGES:
    r = labeler.predict(str(p), conf=0.05, verbose=False)[0]
    h, w = r.orig_shape
    preds = []
    for (x1, y1, x2, y2), c, cf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int),
                                       r.boxes.conf.cpu().numpy()):
        preds.append((int(c), float(cf), (x1 + x2) / 2 / w, (y1 + y2) / 2 / h, (x2 - x1) / w, (y2 - y1) / h))
    CACHE[p.stem] = preds
AUTO_SEC = time.time() - t0
print(f'자동 라벨링: {len(CACHE)}장, {AUTO_SEC:.0f}초 (이미지당 {AUTO_SEC / len(CACHE) * 1000:.0f}ms)')

---
## Step 3. 라벨 품질 검사와 임계값 고르기

### 3-1. 사람 라벨과 맞춰 보기

자동 라벨이 사람 라벨과 얼마나 같은지 클래스별로 셉니다. 같은 클래스이고 **IoU 0.5 이상** 겹치면 '일치'입니다.
- **정밀도**: 자동 라벨 중 사람 라벨과 일치하는 비율 — 낮으면 쓰레기 라벨(헛박스)이 많음
- **재현율**: 사람 라벨 중 자동 라벨이 찾아낸 비율 — 낮으면 빠진 라벨이 많음

신뢰도 기준을 0.1부터 0.6까지 바꿔 가며 F1(정밀도와 재현율의 조화 평균)이 가장 높은 값을 고릅니다.

In [ ]:
def to_xyxy(b):
    cx, cy, w, h = b
    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

def match_counts(auto, human, cls):
    """한 이미지, 한 클래스의 (일치 수, 자동 라벨 수, 사람 라벨 수)"""
    A = [to_xyxy(a[2:]) for a in sorted(auto, key=lambda a: -a[1]) if a[0] == cls]
    Hb = [to_xyxy(h[1:]) for h in human if h[0] == cls]
    used, tp = set(), 0
    for a in A:
        best, j = 0, -1
        for k, hb in enumerate(Hb):
            if k not in used and iou(a, hb) > best:
                best, j = iou(a, hb), k
        if best >= 0.5:
            used.add(j); tp += 1
    return tp, len(A), len(Hb)

HUMAN_TRAIN = {p.stem: read_labels(HUMAN / 'labels' / 'train' / (p.stem + '.txt')) for p in TRAIN_IMAGES}

rows = []
for thr in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    for c in TARGET:
        tp = na = nh = 0
        for stem, preds in CACHE.items():
            t, a, h = match_counts([p for p in preds if p[1] >= thr], HUMAN_TRAIN[stem], c)
            tp += t; na += a; nh += h
        prec, rec = tp / max(na, 1), tp / max(nh, 1)
        rows.append({'conf': thr, 'class': TARGET[c], 'auto_boxes': na, 'precision': round(prec, 3),
                     'recall': round(rec, 3), 'f1': round(2 * prec * rec / max(prec + rec, 1e-9), 3)})
quality = pd.DataFrame(rows)

mean_f1 = quality.groupby('conf')['f1'].mean()
AUTO_CONF = float(mean_f1.idxmax())
print('클래스 평균 F1이 가장 높은 신뢰도 기준:', AUTO_CONF)
quality[quality['conf'] == AUTO_CONF]

In [ ]:
plt.figure(figsize=(8, 3.5))
for c in TARGET.values():
    q = quality[quality['class'] == c]
    plt.plot(q['conf'], q['f1'], '-o', label=c)
plt.axvline(AUTO_CONF, ls='--', c='gray')
plt.xlabel('confidence threshold for auto labels'); plt.ylabel('F1 vs human labels'); plt.ylim(0, 1)
plt.title('Auto-label quality by class'); plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
# 자동 라벨(왼쪽)과 사람 라벨(오른쪽) 나란히 보기 — 차이가 큰 이미지 3장
COLORS = {0: (0, 220, 255), 1: (0, 140, 255), 2: (255, 120, 0)}

def draw_labels(img, labels, has_conf):
    h, w = img.shape[:2]
    for lab in labels:
        c, box = lab[0], lab[2:] if has_conf else lab[1:]
        x1, y1, x2, y2 = (to_xyxy(box) * [w, h, w, h]).astype(int)
        cv2.rectangle(img, (x1, y1), (x2, y2), COLORS[c], 2)
        cv2.putText(img, TARGET[c], (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLORS[c], 2)
    return img

diff = sorted(CACHE, key=lambda s: -abs(len([p for p in CACHE[s] if p[1] >= AUTO_CONF]) - len(HUMAN_TRAIN[s])))[:3]
imgs, titles = [], []
for stem in diff:
    p = next(x for x in TRAIN_IMAGES if x.stem == stem)
    img = cv2.imread(str(p))
    auto = [a for a in CACHE[stem] if a[1] >= AUTO_CONF]
    imgs += [draw_labels(img.copy(), auto, True), draw_labels(img.copy(), HUMAN_TRAIN[stem], False)]
    titles += [f'AUTO {len(auto)} boxes', f'HUMAN {len(HUMAN_TRAIN[stem])} boxes']
show_grid(imgs, titles, cols=2, size=4.5)

**확인해 보기** — 차이가 나는 이유를 분류해 보세요. ① 자동 라벨이 틀린 경우(모자를 안전모로) ② 사람이 빠뜨린 경우(멀리 있는 작은 사람) ③ 기준이 다른 경우(사람 박스를 몸 전체로 그릴지, 보이는 부분만 그릴지). ②와 ③은 자동 라벨의 잘못이 아닙니다.

### 3-2. 자동 라벨 데이터셋 만들기

**학습용 라벨만** 자동 라벨로 바꾸고, 검증(val)·테스트(test) 라벨은 사람 라벨을 그대로 둡니다. 성적은 반드시 사람이 확인한 정답으로 매겨야 하기 때문입니다. 현업에서도 검증용 소량 데이터는 사람이 꼼꼼히 확인해 둡니다.

In [ ]:
def build_dataset(root, train_labels):
    link_images(root)
    for stem, labels in train_labels.items():
        write_labels(root / 'labels' / 'train' / (stem + '.txt'), labels)
    for split in ['val', 'test']:
        d = root / 'labels' / split
        d.mkdir(parents=True, exist_ok=True)
        for f in (HUMAN / 'labels' / split).glob('*.txt'):
            shutil.copy(f, d / f.name)
    return write_yaml(root)

AUTO_LABELS = {s: [(c, *b) for c, cf, *b in preds if cf >= AUTO_CONF] for s, preds in CACHE.items()}
AUTO_YAML = build_dataset(WORK / 'auto', AUTO_LABELS)
n_auto = sum(len(v) for v in AUTO_LABELS.values())
print(f'자동 라벨 박스 {n_auto}개 (사람 라벨 {sum(cnt.values())}개) →', AUTO_YAML)

---
## Step 4. 공장 가동 — 두 모델 학습과 대결

같은 모델(`yolo26n.pt`), 같은 설정, 같은 이미지로 **라벨만 다르게** 두 번 학습합니다. 평가는 둘 다 **사람 라벨 테스트 세트**로 합니다.

In [ ]:
def train_and_test(yaml_path, name):
    m = YOLO('yolo26n.pt')
    t0 = time.time()
    m.train(data=yaml_path, epochs=CFG['epochs'], imgsz=CFG['imgsz'], batch=CFG['batch'],
            project='runs_autolabel', name=name, exist_ok=True, seed=0, verbose=False, plots=False)
    minutes = (time.time() - t0) / 60
    best = YOLO(Path(m.trainer.save_dir) / 'weights' / 'best.pt')
    res = best.val(data=yaml_path, split='test', imgsz=CFG['imgsz'], batch=CFG['batch'], verbose=False, plots=False)
    row = {'labels': name, 'train_min': round(minutes, 1), 'map50': round(res.box.map50, 3),
           'map50_95': round(res.box.map, 3)}
    ap = dict(zip(res.box.ap_class_index.tolist(), res.box.ap50.tolist()))   # 클래스 번호 → AP50
    for i, c in TARGET.items():
        row[f'ap50_{c}'] = round(ap.get(i, 0.0), 3)
    return row

battle = [train_and_test(HUMAN_YAML, 'human'), train_and_test(AUTO_YAML, 'auto')]
pd.DataFrame(battle)

**확인해 보기** — 자동 라벨 모델은 사람 라벨 모델 성능의 몇 %인가요? 클래스별로 보면 어떤 클래스에서 차이가 크게 나나요? Step 3의 클래스별 F1과 연결해 설명해 보세요.

---
## Step 5. 하이브리드 검수와 비용 계산

### 5-1. 애매한 것만 사람이 고친다

자동 라벨 중 **신뢰도가 애매한 박스**(`AUTO_CONF` 이상 `REVIEW_CONF` 미만)가 하나라도 있는 이미지만 사람이 검수한다고 해 봅니다. 여기서는 검수 결과를 '그 이미지의 사람 라벨로 바꾸는 것'으로 흉내 냅니다.

In [ ]:
REVIEW_CONF = 0.6
RUN_HYBRID = True

review = [s for s, preds in CACHE.items() if any(AUTO_CONF <= p[1] < REVIEW_CONF for p in preds)]
print(f'검수 대상 이미지: {len(review)} / {len(CACHE)} ({len(review) / len(CACHE) * 100:.0f}%)')

battle = [b for b in battle if b['labels'] != 'hybrid']   # 이 셀을 다시 실행해도 hybrid 행이 겹치지 않게
if RUN_HYBRID:
    HYB_LABELS = {s: (HUMAN_TRAIN[s] if s in review else AUTO_LABELS[s]) for s in CACHE}
    HYB_YAML = build_dataset(WORK / 'hybrid', HYB_LABELS)
    battle.append(train_and_test(HYB_YAML, 'hybrid'))

battle_df = pd.DataFrame(battle)
battle_df.to_csv('label_battle.csv', index=False); save_out('label_battle.csv')
battle_df

### 5-2. 비용 계산

사람이 라벨링하는 데 걸리는 시간은 현장과 작업자마다 다릅니다. 아래 두 값은 **가정값**이니, 여러분 팀의 경험에 맞게 바꿔서 다시 계산해 보세요.
- `SEC_PER_BOX`: 사람이 박스 하나를 새로 그리는 시간
- `SEC_PER_REVIEW`: 자동 라벨이 달린 이미지 한 장을 검토하고 고치는 시간

In [ ]:
SEC_PER_BOX = 7       # 가정값 — 바꿔 보세요
SEC_PER_REVIEW = 20   # 가정값 — 바꿔 보세요

human_boxes = sum(len(v) for v in HUMAN_TRAIN.values())
cost = pd.DataFrame([
    {'labels': 'human', 'label_hours': round(human_boxes * SEC_PER_BOX / 3600, 2)},
    {'labels': 'auto', 'label_hours': round(AUTO_SEC / 3600, 2)},
    {'labels': 'hybrid', 'label_hours': round((AUTO_SEC + len(review) * SEC_PER_REVIEW) / 3600, 2)},
]).merge(battle_df[['labels', 'map50']], on='labels')
base = cost.loc[cost['labels'] == 'human', 'map50'].iloc[0]
cost['map50_vs_human'] = (cost['map50'] / base).round(3)
cost.to_csv('label_cost.csv', index=False); save_out('label_cost.csv')
cost

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cols = {'human': 'tab:blue', 'auto': 'tab:orange', 'hybrid': 'tab:green'}
axes[0].bar(cost['labels'], cost['label_hours'], color=[cols[l] for l in cost['labels']])
axes[0].set_ylabel('labeling hours (estimate)'); axes[0].set_title('Labeling cost')
for i, v in enumerate(cost['label_hours']):
    axes[0].text(i, v, f'{v:.1f} h', ha='center', va='bottom')
per_class = battle_df.set_index('labels')[[f'ap50_{c}' for c in TARGET.values()]]
per_class.columns = list(TARGET.values())
per_class.T.plot.bar(ax=axes[1], color=[cols[l] for l in per_class.index], rot=0)
axes[1].set_ylim(0, 1); axes[1].set_ylabel('AP50 on human-labeled test set'); axes[1].set_title('Model quality by class')
plt.tight_layout(); plt.savefig('label_report.png', dpi=120); save_out('label_report.png'); plt.show()

결과 파일 (`SAVE_DIR`에 저장)

| 파일 | 내용 |
|---|---|
| `label_battle.csv` | 라벨 방식별 성능 (`labels, train_min, map50, map50_95, ap50_helmet, ap50_vest, ap50_person`) |
| `label_cost.csv` | 라벨 방식별 비용과 성능 (`labels, label_hours, map50, map50_vs_human`) |
| `label_report.png` | 비용·성능 비교 그림 |

---
## 도전 과제

1. **단어 튜닝** — `PROMPTS`의 `'safety vest'`를 `'high-visibility vest'`, `'reflective vest'`로 바꾸면 조끼 라벨 품질(F1)이 올라가나요? 클래스마다 가장 좋은 단어를 찾아 공장을 다시 돌려 보세요.
2. **클래스별 기준** — 지금은 세 클래스에 같은 `AUTO_CONF`를 씁니다. 클래스마다 F1이 가장 높은 기준을 따로 쓰도록 바꿔 보세요.
3. **검수 대상 고르기 전략** — '애매한 신뢰도' 대신 '사람 박스 안에 안전모가 없는 이미지', '박스가 아주 많은 이미지' 등 다른 기준으로 검수 대상을 골라 보세요. 같은 검수 시간에 성능이 더 오르는 전략은 무엇인가요?
4. **내 데이터로** — 캐글이나 공공데이터에서 라벨 없는 이미지 묶음을 구해 자동 라벨 → 학습까지 해 보세요. 성적을 매기려면 검증용 이미지 50~100장 정도는 직접 라벨링해야 합니다.

## 정리

- 오픈 보캐뷸러리 모델은 **라벨링 도구**로도 쓸 수 있습니다. 수천 장 라벨을 몇 분 만에 만듭니다.
- 자동 라벨은 반드시 **사람 라벨과 맞춰 품질을 재고**, 신뢰도 기준도 데이터로 고릅니다.
- 평가는 언제나 **사람이 확인한 정답**으로 합니다. 검증용 소량 데이터는 사람이 만들어야 합니다.
- **애매한 것만 사람이 고치는 하이브리드** 방식이 비용과 성능의 균형점이 되는 경우가 많습니다. 숫자로 따져 보고 결정하세요.